# 02. PU Learning

This notebook treats known business cards as positive examples and consumer cards as unlabeled background.

The goal is not a hard `0/1` classifier. The goal is a business-like ranking for consumer cards, plus enough validation and profile checks to understand why cards were selected.

In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from mdq.data import PROJECT_ROOT, load_data

pd.options.display.float_format = "{:,.4f}".format

RANDOM_STATE = 42
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

## Load Card Features

We reuse the same train/validation/test split from the supervised baseline. Business cards in validation/test are used as hidden positives for recovery checks.

In [2]:
features = pd.read_parquet(PROCESSED_DIR / "card_features_with_split.parquet", engine="fastparquet")
baseline_scores = pd.read_parquet(PROCESSED_DIR / "baseline_scores.parquet", engine="fastparquet")

features.shape, features.groupby(["split", "segment"]).size().unstack(fill_value=0)

((105000, 55),
 segment  business  consumer
 split                      
 test         5000     16000
 train       15000     48000
 val          5000     16000)

In [3]:
ID_COLS = ["card_number", "segment", "label", "split"]

all_feature_cols = [c for c in features.columns if c not in ID_COLS and c != "txn_count"]

mcc_group_cols = [
    c for c in all_feature_cols
    if c.startswith("mcc_") and c.endswith("_ratio")
]

behavior_feature_cols = [c for c in all_feature_cols if c not in mcc_group_cols]

feature_sets = {
    "behavior": behavior_feature_cols,
    "full": all_feature_cols,
}

{name: len(cols) for name, cols in feature_sets.items()}

{'behavior': 34, 'full': 50}

## PU Bagging

PU bagging repeatedly samples a subset of unlabeled consumer cards as temporary negatives, trains a classifier against known business cards, and averages the scores.

This reduces dependence on the false assumption that all consumer cards are true negatives.

In [4]:
def make_linear_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ])


def fit_pu_bagging(data, feature_cols, n_estimators=40, negative_ratio=1.0, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    train = data[data["split"] == "train"]
    positives = train[train["segment"] == "business"]
    unlabeled = train[train["segment"] == "consumer"]

    n_negative = min(len(unlabeled), int(len(positives) * negative_ratio))
    all_scores = []

    for i in range(n_estimators):
        sampled_idx = rng.choice(unlabeled.index.to_numpy(), size=n_negative, replace=False)
        train_idx = np.concatenate([positives.index.to_numpy(), sampled_idx])

        X_train = data.loc[train_idx, feature_cols]
        y_train = (data.loc[train_idx, "segment"] == "business").astype(int)

        model = make_linear_model()
        model.fit(X_train, y_train)
        all_scores.append(model.predict_proba(data[feature_cols])[:, 1])

    score_matrix = np.vstack(all_scores)
    return score_matrix.mean(axis=0), score_matrix.std(axis=0)


pu_bagging_score, pu_bagging_std = fit_pu_bagging(
    features,
    feature_sets["full"],
    n_estimators=40,
    negative_ratio=1.0,
)

features = features.copy()
features["pu_bagging_score"] = pu_bagging_score
features["pu_bagging_std"] = pu_bagging_std

features.groupby("segment")["pu_bagging_score"].describe()

,count,mean,std,min,25%,50%,75%,max
segment,,,,,,,,
business,"25,000.0000",0.9995,0.0089,0.1849,1.0000,1.0000,1.0000,1.0000
consumer,"80,000.0000",0.0007,0.0158,0.0000,0.0000,0.0000,0.0000,0.9960


## Reliable Negative PU

The second PU variant finds consumer cards with very low PU-bagging scores and treats them as reliable negatives. Then it trains a cleaner positive-vs-reliable-negative model.

In [5]:
train = features[features["split"] == "train"]
train_positive = train[train["segment"] == "business"]
train_unlabeled = train[train["segment"] == "consumer"]

reliable_negative_pool = train_unlabeled[train_unlabeled["pu_bagging_score"] <= train_unlabeled["pu_bagging_score"].quantile(0.30)]
n_reliable = min(len(reliable_negative_pool), len(train_positive) * 2)

reliable_negatives = reliable_negative_pool.sort_values("pu_bagging_score").head(n_reliable)

rn_train = pd.concat([train_positive, reliable_negatives], axis=0)
rn_y = (rn_train["segment"] == "business").astype(int)

reliable_negative_model = make_linear_model()
reliable_negative_model.fit(rn_train[feature_sets["full"]], rn_y)

features["reliable_negative_score"] = reliable_negative_model.predict_proba(features[feature_sets["full"]])[:, 1]

pd.Series({
    "train_positives": len(train_positive),
    "reliable_negatives": len(reliable_negatives),
    "reliable_negative_pool_max_score": reliable_negatives["pu_bagging_score"].max(),
})

train_positives                    15,000.0000
reliable_negatives                 14,400.0000
reliable_negative_pool_max_score        0.0000
dtype: float64

## Final PU Rank Score

Raw probabilities from different PU models are not perfectly calibrated. For ranking, percentile ranks are more stable. The final PU score is the average rank percentile from PU bagging and reliable-negative PU.

In [6]:
features["pu_bagging_rank"] = features["pu_bagging_score"].rank(pct=True)
features["reliable_negative_rank"] = features["reliable_negative_score"].rank(pct=True)
features["pu_business_score"] = features[["pu_bagging_rank", "reliable_negative_rank"]].mean(axis=1)

features = features.merge(
    baseline_scores[["card_number", "supervised_business_prob"]],
    on="card_number",
    how="left",
)

features.groupby("segment")[["pu_bagging_score", "reliable_negative_score", "pu_business_score", "supervised_business_prob"]].describe()

pu_bagging_score                                                   \
                    count   mean    std    min    25%    50%    75%    max   
segment                                                                      
business      25,000.0000 0.9995 0.0089 0.1849 1.0000 1.0000 1.0000 1.0000   
consumer      80,000.0000 0.0007 0.0158 0.0000 0.0000 0.0000 0.0000 0.9960   

         reliable_negative_score         ... pu_business_score         \
                           count   mean  ...               75%    max   
segment                                  ...                            
business             25,000.0000 0.9999  ...            0.9344 1.0000   
consumer             80,000.0000 0.0164  ...            0.5662 0.8093   

         supervised_business_prob                                            \
                            count   mean    std    min    25%    50%    75%   
segment                                                                       
business              25,000.0000 0.9996 0.0091 0.1129 1.0000 1.0000 1.0000   
consumer              80,000.0000 0.0006 0.0137 0.0000 0.0000 0.0000 0.0000   

                 
            max  
segment          
business 1.0000  
consumer 0.9975  

[2 rows x 32 columns]

## Held-Out Positive Recovery Metrics

For PU learning, consumer cards in validation/test are unlabeled background, not guaranteed negatives. The useful question is whether held-out business cards are ranked high among this background.

In [7]:
def ranking_metrics(data, split, score_col, k_fracs=(0.001, 0.005, 0.01, 0.05)):
    eval_df = data[data["split"] == split].copy()
    y = eval_df["label"].to_numpy()
    score = eval_df[score_col].to_numpy()

    rows = [{
        "split": split,
        "score": score_col,
        "roc_auc_vs_background": roc_auc_score(y, score),
        "pr_auc_vs_background": average_precision_score(y, score),
        "positive_base_rate": y.mean(),
    }]

    ranked = eval_df.sort_values(score_col, ascending=False).reset_index(drop=True)
    n_pos = ranked["label"].sum()
    base_rate = ranked["label"].mean()

    for frac in k_fracs:
        k = max(1, int(len(ranked) * frac))
        top = ranked.head(k)
        precision_at_k = top["label"].mean()
        recall_at_k = top["label"].sum() / n_pos
        lift_at_k = precision_at_k / base_rate
        rows[0][f"precision_at_{frac:.1%}"] = precision_at_k
        rows[0][f"recall_at_{frac:.1%}"] = recall_at_k
        rows[0][f"lift_at_{frac:.1%}"] = lift_at_k

    return rows[0]


metric_rows = []
for split in ["val", "test"]:
    for score_col in ["pu_bagging_score", "reliable_negative_score", "pu_business_score", "supervised_business_prob"]:
        metric_rows.append(ranking_metrics(features, split, score_col))

pu_validation_metrics = pd.DataFrame(metric_rows)
pu_validation_metrics.to_csv(PROCESSED_DIR / "pu_validation_metrics.csv", index=False)
pu_validation_metrics

,split,score,roc_auc_vs_background,pr_auc_vs_background,positive_base_rate,precision_at_0.1%,recall_at_0.1%,lift_at_0.1%,precision_at_0.5%,recall_at_0.5%,lift_at_0.5%,precision_at_1.0%,recall_at_1.0%,lift_at_1.0%,precision_at_5.0%,recall_at_5.0%,lift_at_5.0%
0,val,pu_bagging_score,1.0000,1.0000,0.2381,1.0000,0.0042,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
1,val,reliable_negative_score,1.0000,1.0000,0.2381,1.0000,0.0042,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
2,val,pu_business_score,1.0000,1.0000,0.2381,1.0000,0.0042,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
3,val,supervised_business_prob,1.0000,1.0000,0.2381,1.0000,0.0042,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
4,test,pu_bagging_score,1.0000,1.0000,0.2381,1.0000,0.0042,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
5,test,reliable_negative_score,1.0000,1.0000,0.2381,1.0000,0.0042,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
6,test,pu_business_score,1.0000,1.0000,0.2381,1.0000,0.0042,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
7,test,supervised_business_prob,1.0000,1.0000,0.2381,1.0000,0.0042,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000


## Consumer Business-Like Tiers

PU scores are ranking scores. We convert them into operational tiers by consumer percentile, not by claiming exact probability calibration.

In [8]:
consumer_scores = features[features["segment"] == "consumer"].copy()
consumer_scores["consumer_score_percentile"] = consumer_scores["pu_business_score"].rank(pct=True)
consumer_scores["business_like_percent"] = consumer_scores["consumer_score_percentile"] * 100

def assign_tier(percentile):
    if percentile >= 0.9995:
        return "tier_1_high_confidence"
    if percentile >= 0.9975:
        return "tier_2_campaign_candidate"
    if percentile >= 0.9900:
        return "tier_3_watchlist"
    return "tier_4_no_action"


consumer_scores["tier"] = consumer_scores["consumer_score_percentile"].apply(assign_tier)

tier_summary = (
    consumer_scores["tier"]
    .value_counts()
    .rename_axis("tier")
    .reset_index(name="cards")
)
tier_summary["share_of_consumer_cards"] = tier_summary["cards"] / len(consumer_scores)
tier_summary.to_csv(PROCESSED_DIR / "pu_tier_summary.csv", index=False)
tier_summary

,tier,cards,share_of_consumer_cards
0,tier_4_no_action,79199,0.9900
1,tier_3_watchlist,600,0.0075
2,tier_2_campaign_candidate,160,0.0020
3,tier_1_high_confidence,41,0.0005


## Human-Readable Reasons

This is a simple reason layer for manual review. It compares candidate cards against known business medians.

In [9]:
reason_cols = [
    "log_total_spend",
    "online_ratio",
    "recurring_ratio",
    "recurring_capable_ratio",
    "foreign_merchant_ratio",
    "weekday_business_hours_ratio",
    "weekend_ratio",
    "evening_ratio",
    "mcc_b2b_total_ratio",
    "mcc_b2b_total_amount_ratio",
]

business_median = features[features["segment"] == "business"][reason_cols].median()

def explain_candidate(row):
    reasons = []
    if row["mcc_b2b_total_amount_ratio"] >= business_median["mcc_b2b_total_amount_ratio"]:
        reasons.append("B2B spend share like business")
    if row["foreign_merchant_ratio"] >= business_median["foreign_merchant_ratio"]:
        reasons.append("high foreign merchant exposure")
    if row["recurring_ratio"] >= business_median["recurring_ratio"]:
        reasons.append("high recurring payments")
    if row["recurring_capable_ratio"] >= business_median["recurring_capable_ratio"]:
        reasons.append("many recurring-capable merchants")
    if row["weekday_business_hours_ratio"] >= business_median["weekday_business_hours_ratio"]:
        reasons.append("work-hours heavy")
    if row["weekend_ratio"] <= business_median["weekend_ratio"]:
        reasons.append("low weekend usage")
    if row["evening_ratio"] <= business_median["evening_ratio"]:
        reasons.append("low evening usage")
    if row["log_total_spend"] >= business_median["log_total_spend"]:
        reasons.append("business-level total spend")
    if row["online_ratio"] >= business_median["online_ratio"]:
        reasons.append("online-heavy")
    return "; ".join(reasons[:5])


consumer_scores["pu_reasons"] = consumer_scores.apply(explain_candidate, axis=1)

score_cols = [
    "card_number",
    "split",
    "pu_business_score",
    "consumer_score_percentile",
    "business_like_percent",
    "tier",
    "pu_reasons",
    "pu_bagging_score",
    "pu_bagging_std",
    "reliable_negative_score",
    "supervised_business_prob",
]

top_candidates = consumer_scores.sort_values("pu_business_score", ascending=False).head(100)
top_candidates[score_cols + reason_cols].head(20)

,card_number,split,pu_business_score,consumer_score_percentile,business_like_percent,tier,pu_reasons,pu_bagging_score,pu_bagging_std,reliable_negative_score,...,log_total_spend,online_ratio,recurring_ratio,recurring_capable_ratio,foreign_merchant_ratio,weekday_business_hours_ratio,weekend_ratio,evening_ratio,mcc_b2b_total_ratio,mcc_b2b_total_amount_ratio
50695,5201491354169846,train,0.8093,1.0000,100.0000,tier_1_high_confidence,high foreign merchant exposure; high recurring...,0.9960,0.0035,0.9999,...,16.4282,0.8846,0.2308,0.4231,0.4423,0.6538,0.0769,0.0769,0.2692,0.3806
67328,5338474007563215,train,0.7647,1.0000,99.9988,tier_1_high_confidence,high foreign merchant exposure; high recurring...,0.8964,0.0930,0.9992,...,16.5119,0.8495,0.1935,0.3548,0.5591,0.6774,0.0968,0.0753,0.4839,0.4629
44522,5176516958585590,test,0.7647,1.0000,99.9975,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.8842,0.0276,0.9992,...,16.8381,0.8986,0.2609,0.8551,0.7536,0.5797,0.1159,0.0580,0.7391,0.8678
37681,5176476691114937,train,0.7647,1.0000,99.9963,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.9422,0.0583,0.9991,...,16.0171,0.8701,0.2338,0.3636,0.3636,0.5714,0.0909,0.1169,0.3506,0.6644
80961,5438816990479651,val,0.7643,1.0000,99.9950,tier_1_high_confidence,work-hours heavy; low weekend usage; low eveni...,0.8736,0.1126,0.9991,...,17.2133,0.7765,0.0000,0.0000,0.0000,0.8588,0.0235,0.0588,0.3294,0.3997
72476,5368297894191789,test,0.7643,0.9999,99.9938,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.7069,0.0752,0.9991,...,17.4043,0.7857,0.3214,0.8393,0.7857,0.5357,0.1250,0.0714,0.8036,0.9247
77969,5438812366882479,train,0.7642,0.9999,99.9925,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.8787,0.1575,0.9990,...,16.3684,0.7778,0.1481,0.7284,0.7407,0.7284,0.0617,0.1111,0.8148,0.8081
95862,5500442485584260,train,0.7641,0.9999,99.9912,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.9259,0.0618,0.9990,...,17.6052,0.8654,0.3462,0.8077,0.8077,0.5000,0.1731,0.0962,0.7115,0.9811
26262,5100612020402608,val,0.7637,0.9999,99.9900,tier_1_high_confidence,high recurring payments; many recurring-capabl...,0.9315,0.0691,0.9988,...,16.3873,0.8276,0.1552,0.3017,0.3017,0.6810,0.1121,0.1121,0.2241,0.5391
53260,5211551692085315,train,0.7636,0.9999,99.9888,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.4250,0.1091,0.9989,...,16.8499,0.9048,0.4286,0.8571,0.8571,0.5000,0.1190,0.0952,0.7143,0.9135


## Transaction-Level Profile For Top Candidates

To manually inspect selected cards, we return to transaction-level data and summarize top merchants and MCC categories for the highest-ranked consumer cards.

In [10]:
business_raw, consumer_raw, merchants, mcc = load_data()

top_card_numbers = top_candidates["card_number"].head(50).tolist()
top_txn = consumer_raw[consumer_raw["card_number"].isin(top_card_numbers)].copy()
top_txn["mcc"] = top_txn["mcc"].astype(str)

merchants_no_mcc = merchants.drop(columns=["mcc"]).copy()
top_txn = top_txn.merge(merchants_no_mcc, on="merchant_id", how="left")

mcc_ref = mcc[["mcc", "mcc_name"]].copy()
mcc_ref["mcc"] = mcc_ref["mcc"].astype(str)
top_txn = top_txn.merge(mcc_ref, on="mcc", how="left")

mask = (top_txn["merchant_id"] == "MER_000000") & (top_txn["mcc"] == "7012")
top_txn.loc[mask, "merchant_name"] = "MER_000000 / MCC 7012 placeholder"

top_txn["ts"] = pd.to_datetime(top_txn["transaction_timestamp"])
top_txn["is_online"] = top_txn["channel"].str.lower().eq("online")
top_txn["is_weekend"] = top_txn["ts"].dt.dayofweek >= 5
top_txn["weekday_business_hours"] = (~top_txn["is_weekend"]) & top_txn["ts"].dt.hour.between(9, 18)
top_txn["foreign_merchant"] = top_txn["merchant_country"].ne("Kazakhstan")

b2b_codes = {"7311", "7372", "7399", "7392", "5943", "5111", "4814", "4816", "4214", "4215", "4225", "8111", "8931", "8999"}
top_txn["is_b2b_mcc"] = top_txn["mcc"].isin(b2b_codes)

def top_values(series, n=5):
    return "; ".join([f"{idx}: {val}" for idx, val in series.value_counts().head(n).items()])

txn_profile = top_txn.groupby("card_number").agg(
    txn_count=("transaction_amount_kzt", "size"),
    total_spend=("transaction_amount_kzt", "sum"),
    avg_amount=("transaction_amount_kzt", "mean"),
    online_ratio=("is_online", "mean"),
    recurring_ratio=("is_recurring", "mean"),
    foreign_merchant_ratio=("foreign_merchant", "mean"),
    weekday_business_hours_ratio=("weekday_business_hours", "mean"),
    b2b_mcc_ratio=("is_b2b_mcc", "mean"),
    top_merchants=("merchant_name", top_values),
    top_mcc_names=("mcc_name", top_values),
).reset_index()

top_candidate_profiles = top_candidates[score_cols + reason_cols].merge(txn_profile, on="card_number", how="left")
top_candidate_profiles.to_csv(PROCESSED_DIR / "pu_top_candidate_transaction_profiles.csv", index=False)

top_candidate_profiles.head(20)

,card_number,split,pu_business_score,consumer_score_percentile,business_like_percent,tier,pu_reasons,pu_bagging_score,pu_bagging_std,reliable_negative_score,...,txn_count,total_spend,avg_amount,online_ratio_y,recurring_ratio_y,foreign_merchant_ratio_y,weekday_business_hours_ratio_y,b2b_mcc_ratio,top_merchants,top_mcc_names
0,5201491354169846,train,0.8093,1.0000,100.0000,tier_1_high_confidence,high foreign merchant exposure; high recurring...,0.9960,0.0035,0.9999,...,52.0000,"13,635,876.0000","262,228.3846",0.8846,0.2308,0.4423,0.6538,0.2692,CommuterTransport_1921: 17; HubSpot: 14; Canva...,Transportation: Suburban and Local Commuter Pa...
1,5338474007563215,train,0.7647,1.0000,99.9988,tier_1_high_confidence,high foreign merchant exposure; high recurring...,0.8964,0.0930,0.9992,...,93.0000,"14,825,660.0000","159,415.6989",0.8495,0.1935,0.5591,0.6774,0.4839,DurableGoods_9947: 29; FedEx: 19; Salesforce: ...,Durable Goods: not elsewhere classified: 29; C...
2,5176516958585590,test,0.7647,1.0000,99.9975,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.8842,0.0276,0.9992,...,69.0000,"20,545,012.0000","297,753.7971",0.8986,0.2609,0.7536,0.5797,0.7391,Instagram Promote: 37; Notion: 8; Children's/I...,Advertising Services: 37; Direct Marketing: Co...
3,5176476691114937,train,0.7647,1.0000,99.9963,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.9422,0.0583,0.9991,...,77.0000,"9,039,253.0000","117,392.8961",0.8701,0.2338,0.3636,0.5714,0.3506,PackageStores-Bee_3021: 27; Atlassian: 11; GoD...,"Package Stores, Beer, Wine, and Liquor: 27; Co..."
4,5438816990479651,val,0.7643,1.0000,99.9950,tier_1_high_confidence,work-hours heavy; low weekend usage; low eveni...,0.8736,0.1126,0.9991,...,85.0000,"29,896,519.0000","351,723.7529",0.7765,0.0000,0.0000,0.8588,0.3294,FamilyClothingSt_6936: 33; Pony Express KZ: 19...,Family Clothing Stores: 33; Courier Services: ...
5,5368297894191789,test,0.7643,0.9999,99.9938,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.7069,0.0752,0.9991,...,56.0000,"36,188,922.0000","646,230.7500",0.7857,0.3214,0.7857,0.5357,0.8036,TikTok Ads: 25; Microsoft Azure: 9; ComputerNe...,Advertising Services: 25; Computer Programming...
6,5438812366882479,train,0.7642,0.9999,99.9925,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.8787,0.1575,0.9990,...,81.0000,"12,843,861.0000","158,566.1852",0.7778,0.1481,0.7407,0.7284,0.8148,GoDaddy: 36; Amazon Web Services: 23; DurableG...,Computer Network/Information Services: 36; Com...
7,5500442485584260,train,0.7641,0.9999,99.9912,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.9259,0.0618,0.9990,...,52.0000,"44,244,475.0000","850,855.2885",0.8654,0.3462,0.8077,0.5000,0.7115,Yandex Direct: 30; Taxicabs/Limousine_8351: 7;...,Advertising Services: 30; Limousines and Taxic...
8,5100612020402608,val,0.7637,0.9999,99.9900,tier_1_high_confidence,high recurring payments; many recurring-capabl...,0.9315,0.0691,0.9988,...,116.0000,"13,089,362.0000","112,839.3276",0.8276,0.1552,0.3017,0.6810,0.2241,PackageStores-Bee_3840: 44; Hotel/Lodging(Cod_...,"Package Stores, Beer, Wine, and Liquor: 44; Lo..."
9,5211551692085315,train,0.7636,0.9999,99.9888,tier_1_high_confidence,B2B spend share like business; high foreign me...,0.4250,0.1091,0.9989,...,42.0000,"20,788,658.0000","494,968.0476",0.9048,0.4286,0.8571,0.5000,0.7143,Instagram Promote: 18; Salesforce: 6; Hetzner:...,Advertising Services: 18; Computer Programming...


## Save PU Outputs

In [11]:
pu_scores = features[[
    "card_number",
    "segment",
    "label",
    "split",
    "pu_bagging_score",
    "pu_bagging_std",
    "reliable_negative_score",
    "pu_business_score",
    "supervised_business_prob",
]].copy()

pu_scores.to_parquet(PROCESSED_DIR / "pu_scores.parquet", index=False, engine="fastparquet")

consumer_scores[score_cols + reason_cols].to_parquet(PROCESSED_DIR / "pu_consumer_scores.parquet", index=False, engine="fastparquet")
top_candidates[score_cols + reason_cols].to_csv(PROCESSED_DIR / "pu_top_candidates.csv", index=False)

pd.Series({
    "pu_scores_rows": len(pu_scores),
    "consumer_scores_rows": len(consumer_scores),
    "top_candidates_rows": len(top_candidates),
})

pu_scores_rows          105000
consumer_scores_rows     80000
top_candidates_rows        100
dtype: int64

## What Happens Under The Hood

- PU Bagging does not trust all consumer cards as negatives. It samples many different temporary negative sets from consumer cards and averages the resulting scores.
- Reliable-negative PU uses the lowest-scoring consumer cards as a cleaner negative set, then learns a positive-vs-reliable-negative separator.
- The final PU score is a rank ensemble of the two PU variants.
- Consumer tiers are percentiles, not calibrated probabilities.
- Transaction-level profiles are used only for manual inspection and business explanation after scoring.